# Step 4 — Deterministic cleaning and audit log

**Purpose.** Apply the documented rules from `step4_decision_table.md` to the three raw MyAnimeList tables, without changing the raw files. The result is a versioned processed dataset for Steps 5–6.

**Success criteria.** This notebook is complete when it writes cleaned anime, user and chunked list tables; records each row-count change and value-level rule; and verifies the expected number of duplicate-account rows removed. It is deliberately not an analysis notebook.

**Important design choices.** Raw columns are retained unless a column is entirely empty (`access_rank`). Corrected values go in `*_clean` columns and every important correction has a flag. The Bayesian score is not calculated here: its data-derived constants must be fitted inside an ML pipeline if it is used as an ML feature.


## Rules implemented

- Safe CSV loading preserves the username `null`; empty strings alone are missing.
- Remove only `Io_Otonashi` and `Dojin_`: duplicate aliases with identical lists.
- Convert documented placeholders and sentinel values to missing only in cleaned columns.
- Retain partial list-date years when they are unambiguously inside 2005–2018; retain full dates only from 2004-11-01 through 2018-05-25.
- Treat birth year 1930 and ages outside 13–55 at 2018-01-01 as suspect; preserve raw birth date and expose only `birth_year_clean` for age analyses.
- Keep extreme activity totals raw and flag them; analysis-safe copies set only the documented impossible maxima to missing.


## What this notebook did, in plain words

**Purpose.** Steps 1 to 3 only looked at the raw data and listed the problems. This notebook applies the cleaning rules we agreed on in the decision table. It changes **no raw file**: every correction goes into a new `*_clean` column, and every removal is written to a log.

**The five principles.**
1. **Keep the raw value and add a clean value beside it.** Anyone can compare the two, and a robustness check (D6) can change a rule without redoing the work.
2. **Deterministic fixes only.** A rule such as "`score = 0` means no score" gives the same result on any data. Anything that has to be learned from the data (a mean, a percentile, a scaler) is left for the scikit-learn `Pipeline`, fitted on the training set only.
3. **Flag instead of delete.** Suspicious rows (few raters, extreme users, doubtful birth dates) stay in the table with a flag, and each question decides its own filter and reports its own row count.
4. **Log everything.** `row_count_log.csv` records every change in the number of rows, and `value_cleaning_log.csv` records how many values were made missing or flagged by each rule.
5. **Test first.** With `TEST_MODE = True` only two small chunks of the list table are processed, so the rules can be checked in a couple of minutes before the full run.

**What was done to each table.** The counts come from the step 3 audit on raw data. The counts of this run are in the two log files.

| Table | Rows removed | What the cleaning does |
|---|---|---|
| `AnimeList` | 0 | Words that mean "no value" (`Unknown`, `None`, `Not available`, `?`) become missing. `score`, `rank`, `popularity` and `episodes` equal to 0 become missing. `duration` is converted to minutes (313 anime wrote it in seconds). The release year is read from `aired_string`, because the dates in `aired` are partly made up. HTML codes are decoded. `low_raters` flags the anime with fewer than 100 raters. |
| `UserList` | 2 | The two duplicate accounts `Io_Otonashi` and `Dojin_` are removed (their lists are identical to `Edengarde` and `DXJIN`). `access_rank` (always empty) is dropped. Birth year 1930 and ages outside 13–55 are marked as suspect, and only a clean birth year is exposed. `last_online` of 1900 and `stats_mean_score = 0` become missing. Impossible activity totals become missing, and extreme users are flagged. |
| `UserAnimeList` | 1,059 | The rows of the two duplicate accounts are removed. Unknown statuses (537 rows), `score = 0`, the sentinels 65,535 and 16,777,215, `0000-00-00` dates, dates outside 2004-11-01 to 2018-05-25 and `my_last_updated = 0` become missing. Where a user watched more episodes than the anime has, the episode count becomes missing. Tags are cleaned and de-duplicated. |

**Expected row counts, as a check of the run.** Users 302,675 → 302,673. List rows 80,076,112 → 80,075,053. Anime 14,478 → 14,478. The asserts in the notebook stop the run if a count is different.

**What is saved.** The anime and user tables are saved as CSV with all raw and clean columns and flags. The list table is saved in parts as **parquet** and keeps only the 13 cleaned columns that the analyses use. The redundant flags of the list table are not stored, because each can be rebuilt from the raw and clean columns and its count is in the log.


In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import html
import re
import shutil
import sys

import numpy as np
import pandas as pd
import pyarrow  # needed for parquet output of the list table

KAGGLE_INPUT = Path('/kaggle/input')
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'environment.yml').exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
    PROCESSED_ROOT = Path('/kaggle/working/processed')
elif ROOT is not None:
    DATA_DIR = ROOT / 'data' / 'raw'
    PROCESSED_ROOT = ROOT / 'data' / 'processed'
else:
    raise RuntimeError('Data not found: run on Kaggle or inside the repository.')

paths = {p.name: p for p in DATA_DIR.rglob('*.csv')}
required = {'AnimeList.csv', 'UserList.csv', 'UserAnimeList.csv'}
missing_files = required - paths.keys()
if missing_files:
    raise FileNotFoundError(f'Missing raw files: {sorted(missing_files)}')

TEST_MODE = False  # True: only the first 2 small list chunks, written to a separate '_test' folder
VERSION = 'step4_clean_v1' + ('_test' if TEST_MODE else '')
OUTPUT_DIR = PROCESSED_ROOT / VERSION
LIST_DIR = OUTPUT_DIR / 'user_anime_list_parts'
OVERWRITE_OUTPUT = False  # Set True only to replace this notebook's previous output.
CHUNK_ROWS = 200_000 if TEST_MODE else 3_000_000
CRAWL_END = pd.Timestamp('2018-05-25')
SITE_START = pd.Timestamp('2004-11-01')
AGE_REFERENCE = pd.Timestamp('2018-01-01')
AGE_MIN, AGE_MAX = 13, 55
ALIAS_TO_DROP = {'Io_Otonashi', 'Dojin_'}
KNOWN_STATUS = {1, 2, 3, 4, 6}
na_kwargs = dict(keep_default_na=False, na_values=[''])

if OUTPUT_DIR.exists():
    if not OVERWRITE_OUTPUT:
        raise FileExistsError(f'{OUTPUT_DIR} already exists. Inspect it or set OVERWRITE_OUTPUT = True deliberately.')
    shutil.rmtree(OUTPUT_DIR)
LIST_DIR.mkdir(parents=True, exist_ok=False)

print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print('Reading raw data from:', DATA_DIR)
print('Writing processed data to:', OUTPUT_DIR)


Python: 3.13.18 | pandas: 2.3.3
Reading raw data from: /kaggle/input
Writing processed data to: /kaggle/working/processed/step4_clean_v1


In [2]:
# Small helpers. `*_clean` values are deterministic transformations of raw values.
PLACEHOLDER_RE = re.compile(
    r'^\s*(?:|\?+|-+|\.+|_+|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$',
    flags=re.IGNORECASE,
)
# A full date has a real month (01-12) and day (01-31). Partial dates such as 2012-00-00 are NOT full.
FULL_DATE_RE = re.compile(r'^\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])$')
YEAR_DATE_RE = re.compile(r'^(\d{4})-(?:\d{2}|00)-(?:\d{2}|00)$')

row_log: list[dict] = []
value_log: Counter = Counter()

def log_rows(table: str, rule: str, before: int, after: int, reason: str) -> None:
    row_log.append({'table': table, 'rule': rule, 'before_rows': before, 'after_rows': after,
                    'row_delta': after - before, 'reason': reason})
    print(f'{table} | {rule}: {before:,} -> {after:,} rows ({after - before:+,})')

def text_clean(value: object, placeholders: bool = True) -> object:
    if pd.isna(value):
        return pd.NA
    value = html.unescape(str(value)).strip()
    if not value:
        return pd.NA
    if placeholders and PLACEHOLDER_RE.fullmatch(value):
        return pd.NA
    return value

def clean_text_series(series: pd.Series, placeholders: bool = True) -> pd.Series:
    return series.map(lambda v: text_clean(v, placeholders)).astype('string')

def list_date_columns(series: pd.Series, prefix: str) -> pd.DataFrame:
    """Keep a full valid date and, separately, a safe year from full/partial date text."""
    raw = series.astype('string')
    full = raw.str.fullmatch(FULL_DATE_RE.pattern, na=False)
    parsed = pd.to_datetime(raw.where(full), format='%Y-%m-%d', errors='coerce')
    date_ok = parsed.ge(SITE_START) & parsed.le(CRAWL_END)
    year = pd.to_numeric(raw.str.extract(YEAR_DATE_RE.pattern, expand=False), errors='coerce').astype('Int64')
    # A partial 2004 date could precede the site's launch, so it is not safe enough to retain.
    year_ok = year.between(2005, CRAWL_END.year) | (full & date_ok)
    out = pd.DataFrame(index=series.index)
    out[f'{prefix}_date_clean'] = parsed.where(date_ok)
    out[f'{prefix}_year_clean'] = year.where(year_ok)
    out[f'{prefix}_date_invalid'] = raw.notna() & ~raw.eq('0000-00-00') & ~date_ok & full
    out[f'{prefix}_year_partial'] = year.gt(0).fillna(False) & ~full
    return out

def clean_tags(value: object) -> object:
    if pd.isna(value):
        return pd.NA
    tags, seen = [], set()
    for part in html.unescape(str(value)).split(','):
        tag = part.strip().lower()
        if not tag or PLACEHOLDER_RE.fullmatch(tag) or tag in seen:
            continue
        seen.add(tag)
        tags.append(tag)
    return ', '.join(tags) if tags else pd.NA


## A. Anime table

The original text columns remain. Their cleaned counterparts decode HTML, trim spaces, and replace documented placeholder words with missing values. Empty `related` is labelled `none`; empty theme lists are missing. Release dates are read from the text only when it is complete, avoiding the known made-up components in `aired`.


In [3]:
anime = pd.read_csv(paths['AnimeList.csv'], **na_kwargs)
anime_before = len(anime)

anime_text_placeholders = 0
# Free-text columns: decode HTML and trim only. A real title such as 'None' must not be blanked.
FREE_TEXT = {'title', 'title_english', 'title_japanese', 'title_synonyms', 'background', 'image_url'}
for col in anime.select_dtypes(include=['object', 'string']).columns:
    cleaned = clean_text_series(anime[col], placeholders=col not in FREE_TEXT)
    anime_text_placeholders += int(anime[col].notna().sum() - cleaned.notna().sum())
    anime[f'{col}_clean'] = cleaned

anime['related_clean'] = anime['related_clean'].mask(anime['related'].eq('[]'), 'none')
anime['aired_clean'] = anime['aired_clean'].mask(anime['aired'].eq("{'from': None, 'to': None}"))
for col in ['opening_theme_clean', 'ending_theme_clean']:
    anime[col] = anime[col].mask(anime[col].eq('[]'))

anime['score_clean'] = anime['score'].where(anime['score'].ne(0))
anime['episodes_clean'] = anime['episodes'].where(anime['episodes'].gt(0)).astype('Int64')
anime['rank_clean'] = anime['rank'].where(anime['rank'].gt(0))
anime['popularity_clean'] = anime['popularity'].where(anime['popularity'].gt(0)).astype('Int64')
anime['low_raters'] = anime['scored_by'].lt(100)

duration = anime['duration_clean'].fillna('')
hours = pd.to_numeric(duration.str.extract(r'(\d+(?:\.\d+)?)\s*hr', expand=False), errors='coerce').fillna(0)
minutes = pd.to_numeric(duration.str.extract(r'(\d+(?:\.\d+)?)\s*min', expand=False), errors='coerce').fillna(0)
seconds = pd.to_numeric(duration.str.extract(r'(\d+(?:\.\d+)?)\s*sec', expand=False), errors='coerce').fillna(0)
has_duration = duration.str.contains(r'\d+(?:\.\d+)?\s*(?:hr|min|sec)', regex=True)
anime['duration_minutes_clean'] = (hours * 60 + minutes + seconds / 60).where(has_duration)

parts = anime['aired_string_clean'].fillna('').str.split(r'\s+to\s+', n=1, expand=True)
start_text = parts[0]
end_text = parts[1] if 1 in parts.columns else pd.Series(pd.NA, index=anime.index, dtype='string')
for name, text in [('aired_start', start_text), ('aired_end', end_text)]:
    full = text.str.fullmatch(r'[A-Z][a-z]{2} \d{1,2}, \d{4}', na=False)
    parsed = pd.to_datetime(text.where(full), format='%b %d, %Y', errors='coerce')
    anime[f'{name}_date_clean'] = parsed
    anime[f'{name}_year_clean'] = pd.to_numeric(text.str.extract(r'(\d{4})', expand=False), errors='coerce').astype('Int64')
    anime[f'{name}_partial_text'] = text.notna() & text.ne('') & ~full & text.ne('?')

anime['not_yet_aired_with_completed_rows'] = False  # Filled after the chunked list pass.
value_log.update({
    'anime.text_placeholder_to_missing': anime_text_placeholders,
    'anime.score_zero_to_missing': int(anime['score'].eq(0).sum()),
    'anime.episodes_zero_to_missing': int(anime['episodes'].eq(0).sum()),
    'anime.rank_zero_to_missing': int(anime['rank'].eq(0).sum()),
    'anime.popularity_zero_to_missing': int(anime['popularity'].eq(0).sum()),
    'anime.low_raters': int(anime['low_raters'].sum()),
})
log_rows('anime', 'value corrections only', anime_before, len(anime), 'No anime rows are removed; raw values are retained beside cleaned columns.')


anime | value corrections only: 14,478 -> 14,478 rows (+0)


## B. User table

The two confirmed duplicate aliases are removed here and again during list processing. No user is removed for missing demographics, a suspect birth date, an extreme total, or an empty profile.


In [4]:
users = pd.read_csv(paths['UserList.csv'], **na_kwargs)
users_before = len(users)
if not users['username'].is_unique:
    raise ValueError('username must be unique before alias removal.')

alias_rows = users['username'].isin(ALIAS_TO_DROP)
if set(users.loc[alias_rows, 'username']) != ALIAS_TO_DROP:
    raise ValueError('Expected duplicate aliases were not found; stop and inspect the source data.')
users = users.loc[~alias_rows].copy()
log_rows('users', 'remove duplicate aliases', users_before, len(users),
         'Io_Otonashi and Dojin_ duplicate Edengarde and DXJIN respectively, including identical lists.')

users = users.drop(columns=['access_rank'])
users['gender_clean'] = clean_text_series(users['gender'])
users['location_clean'] = clean_text_series(users['location']).str.lower().str.replace(r'\s+', ' ', regex=True)

birth = pd.to_datetime(users['birth_date'], errors='coerce')
age = (AGE_REFERENCE - birth).dt.days / 365.25
users['birth_date_parse_failed'] = users['birth_date'].notna() & birth.isna()
users['birth_year_1930_default'] = birth.dt.year.eq(1930)
users['birth_age_suspect'] = birth.notna() & (users['birth_year_1930_default'] | age.lt(AGE_MIN) | age.gt(AGE_MAX))
users['birth_year_clean'] = birth.dt.year.where(~users['birth_age_suspect']).astype('Int64')

join = pd.to_datetime(users['join_date'], errors='coerce')
online = pd.to_datetime(users['last_online'], errors='coerce')
users['join_date_clean'] = join
users['last_online_1900_default'] = online.dt.year.eq(1900)
users['last_online_clean'] = online.where(~users['last_online_1900_default'])
users['account_days_at_crawl'] = (CRAWL_END - join).dt.days

users['stats_mean_score_clean'] = users['stats_mean_score'].where(users['stats_mean_score'].gt(0))
users['extreme_user'] = users['user_days_spent_watching'].gt(1_000) | users['stats_episodes'].gt(100_000)
users['days_watched_impossible'] = users['user_days_spent_watching'].gt(10_000)
users['stats_episodes_impossible'] = users['stats_episodes'].gt(500_000)
users['user_days_spent_watching_clean'] = users['user_days_spent_watching'].where(~users['days_watched_impossible'])
users['stats_episodes_clean'] = users['stats_episodes'].where(~users['stats_episodes_impossible'])
users['watched_days_exceed_account_days'] = users['user_days_spent_watching'].gt(users['account_days_at_crawl'])
users['zero_completed_millions_episodes'] = users['user_completed'].eq(0) & users['stats_episodes'].ge(1_000_000)
users['has_list_rows'] = False  # Updated after the chunked list pass.

value_log.update({
    'users.access_rank_dropped': 1,
    'users.birth_date_suspect': int(users['birth_age_suspect'].sum()),
    'users.last_online_1900_to_missing': int(users['last_online_1900_default'].sum()),
    'users.stats_mean_score_zero_to_missing': int(users['stats_mean_score'].eq(0).sum()),
    'users.extreme_user_flag': int(users['extreme_user'].sum()),
    'users.days_watched_impossible_to_missing': int(users['days_watched_impossible'].sum()),
    'users.stats_episodes_impossible_to_missing': int(users['stats_episodes_impossible'].sum()),
})
assert len(users) == 302_673, f'Unexpected cleaned user count: {len(users):,}'


users | remove duplicate aliases: 302,675 -> 302,673 rows (-2)


## C. List table — chunked processing

The 80-million-row file is never loaded at once. Each part is cleaned and written immediately as **parquet**, which is much smaller than CSV. To save space, the output keeps only the cleaned columns that the analyses use (`LIST_KEEP`). The raw file stays the source for any raw value, and the redundant flags (episodes inconsistent, completed count inferred, invalid or partial date, unreliable viewing dates) are **not stored**: each can be re-derived from the raw and clean columns, and its count is kept in `value_cleaning_log.csv`.


In [5]:
# Use cleaned episode counts for consistency checks.
episodes_by_anime = anime.set_index('anime_id')['episodes_clean']
unaired_ids = set(anime.loc[anime['status'].eq('Not yet aired'), 'anime_id'])
completed_on_unaired: set[int] = set()
users_with_rows: set[str] = set()
raw_list_rows = clean_list_rows = alias_list_rows = 0

LIST_KEEP = [
    'username', 'anime_id', 'my_status_clean', 'my_score_clean', 'my_watched_episodes_clean',
    'my_start_date_clean', 'my_finish_date_clean', 'my_start_year_clean', 'my_finish_year_clean',
    'my_last_updated_clean', 'my_rewatching_state', 'my_rewatching_ep_clean', 'my_tags_clean',
]
list_usecols = [
    'username', 'anime_id', 'my_watched_episodes', 'my_start_date', 'my_finish_date',
    'my_score', 'my_status', 'my_rewatching', 'my_rewatching_ep', 'my_last_updated', 'my_tags',
]
list_dtypes = {
    'username': 'string', 'anime_id': 'int32', 'my_watched_episodes': 'Int32',
    'my_score': 'Int8', 'my_status': 'Int16', 'my_rewatching': 'Int8',
    'my_rewatching_ep': 'Int32', 'my_last_updated': 'Int64',
}

reader = pd.read_csv(paths['UserAnimeList.csv'], usecols=list_usecols, dtype=list_dtypes,
                     chunksize=CHUNK_ROWS, **na_kwargs)
for part, chunk in enumerate(reader, start=1):
    if TEST_MODE and part > 2:
        break
    raw_list_rows += len(chunk)
    aliases = chunk['username'].isin(ALIAS_TO_DROP)
    alias_list_rows += int(aliases.sum())
    chunk = chunk.loc[~aliases].copy()
    clean_list_rows += len(chunk)
    users_with_rows.update(chunk['username'].dropna().unique().tolist())

    unknown_status = ~chunk['my_status'].isin(KNOWN_STATUS)
    chunk['my_status_clean'] = chunk['my_status'].where(~unknown_status, pd.NA).astype('Int8')
    chunk['my_score_clean'] = chunk['my_score'].where(chunk['my_score'].gt(0), pd.NA).astype('Int8')
    value_log['lists.unknown_status_to_missing'] += int(unknown_status.sum())

    start = list_date_columns(chunk['my_start_date'], 'my_start')
    finish = list_date_columns(chunk['my_finish_date'], 'my_finish')
    chunk = pd.concat([chunk, start, finish], axis=1)
    bad_order = (chunk['my_finish_date_clean'] < chunk['my_start_date_clean']).fillna(False)
    chunk['viewing_dates_unreliable'] = bad_order
    chunk.loc[bad_order, ['my_start_date_clean', 'my_finish_date_clean']] = pd.NaT
    value_log['lists.date_full_invalid_to_missing'] += int(start['my_start_date_invalid'].sum() + finish['my_finish_date_invalid'].sum())
    value_log['lists.finish_before_start_flag'] += int(bad_order.sum())
    value_log['lists.date_year_partial'] += int(start['my_start_year_partial'].sum() + finish['my_finish_year_partial'].sum())

    updated = pd.to_datetime(chunk['my_last_updated'], unit='s', errors='coerce')
    updated_ok = chunk['my_last_updated'].gt(0) & updated.ge(SITE_START) & updated.le(CRAWL_END + pd.Timedelta(days=1))
    chunk['my_last_updated_clean'] = updated.where(updated_ok)
    value_log['lists.last_updated_invalid_to_missing'] += int((~updated_ok).sum())

    watched_sentinel = chunk['my_watched_episodes'].eq(65_535)
    watched = chunk['my_watched_episodes'].where(~watched_sentinel, pd.NA).astype('Int32')
    total = chunk['anime_id'].map(episodes_by_anime).astype('Int64')
    watched_over = (total.notna() & watched.notna() & watched.gt(total)).fillna(False)
    chunk['episodes_inconsistent'] = watched_over
    chunk['my_watched_episodes_clean'] = watched.where(~watched_over, pd.NA).astype('Int32')
    completed_less = (chunk['my_status_clean'].eq(2) & total.notna() & (watched.isna() | watched.lt(total))).fillna(False)
    chunk['my_watched_episodes_completed_fill'] = chunk['my_watched_episodes_clean'].where(~completed_less, total).astype('Int32')
    chunk['completed_count_inferred'] = completed_less
    value_log['lists.watched_episode_sentinel_to_missing'] += int(watched_sentinel.sum())
    value_log['lists.watched_over_anime_episodes_to_missing'] += int(watched_over.sum())
    value_log['lists.completed_episode_count_inferred'] += int(completed_less.sum())

    rewatch_sentinel = chunk['my_rewatching_ep'].eq(16_777_215)
    chunk['my_rewatching_ep_clean'] = chunk['my_rewatching_ep'].where(~rewatch_sentinel, pd.NA).astype('Int32')
    chunk['my_rewatching_state'] = np.select(
        [chunk['my_rewatching'].eq(1).fillna(False).to_numpy(dtype=bool), chunk['my_rewatching'].eq(0).fillna(False).to_numpy(dtype=bool)], ['yes', 'no'], default='unknown'
    )
    value_log['lists.rewatch_episode_sentinel_to_missing'] += int(rewatch_sentinel.sum())

    tags_clean = chunk['my_tags'].map(clean_tags).astype('string')
    chunk['my_tags_clean'] = tags_clean
    value_log['lists.tag_placeholder_or_empty_to_missing'] += int(chunk['my_tags'].notna().sum() - tags_clean.notna().sum())

    unaired_completed = chunk['my_status_clean'].eq(2) & chunk['anime_id'].isin(unaired_ids)
    completed_on_unaired.update(chunk.loc[unaired_completed, 'anime_id'].unique().tolist())

    part_path = LIST_DIR / f'UserAnimeList_clean_{part:03d}.parquet'
    chunk[LIST_KEEP].to_parquet(part_path, index=False)
    print(f'part {part:03d}: raw {raw_list_rows:,}; kept {clean_list_rows:,}; wrote {part_path.name}')

if not TEST_MODE:  # the full-file totals only make sense on a full run
    assert raw_list_rows == 80_076_112, f'Unexpected raw list row count: {raw_list_rows:,}'
    assert alias_list_rows == 1_059, f'Expected 1,059 duplicate alias rows, found {alias_list_rows:,}'
    assert clean_list_rows == 80_075_053, f'Unexpected cleaned list row count: {clean_list_rows:,}'
log_rows('lists', 'remove duplicate alias rows', raw_list_rows, clean_list_rows,
         'The lists for two renamed duplicate accounts are exact duplicates of the retained accounts.')


part 001: raw 3,000,000; kept 2,999,478; wrote UserAnimeList_clean_001.parquet
part 002: raw 6,000,000; kept 5,999,478; wrote UserAnimeList_clean_002.parquet
part 003: raw 9,000,000; kept 8,998,941; wrote UserAnimeList_clean_003.parquet
part 004: raw 12,000,000; kept 11,998,941; wrote UserAnimeList_clean_004.parquet
part 005: raw 15,000,000; kept 14,998,941; wrote UserAnimeList_clean_005.parquet
part 006: raw 18,000,000; kept 17,998,941; wrote UserAnimeList_clean_006.parquet
part 007: raw 21,000,000; kept 20,998,941; wrote UserAnimeList_clean_007.parquet
part 008: raw 24,000,000; kept 23,998,941; wrote UserAnimeList_clean_008.parquet
part 009: raw 27,000,000; kept 26,998,941; wrote UserAnimeList_clean_009.parquet
part 010: raw 30,000,000; kept 29,998,941; wrote UserAnimeList_clean_010.parquet
part 011: raw 33,000,000; kept 32,998,941; wrote UserAnimeList_clean_011.parquet
part 012: raw 36,000,000; kept 35,998,941; wrote UserAnimeList_clean_012.parquet
part 013: raw 39,000,000; kept 38,

## D. Finalise, validate and write the audit log

`has_list_rows` and the six not-yet-aired flags depend on the complete list pass, so they are set only now. The log records row removal separately from values made missing or flagged.


In [6]:
users['has_list_rows'] = users['username'].isin(users_with_rows)
unmatched = users_with_rows - set(users['username'])
assert not unmatched, f'{len(unmatched):,} list usernames are missing from the user table, e.g. {sorted(unmatched)[:5]}'
value_log['lists.usernames_not_in_user_table'] = len(unmatched)
anime['not_yet_aired_with_completed_rows'] = anime['anime_id'].isin(completed_on_unaired)

anime_path = OUTPUT_DIR / 'AnimeList_clean.csv'
users_path = OUTPUT_DIR / 'UserList_clean.csv'
anime.to_csv(anime_path, index=False)
users.to_csv(users_path, index=False)

value_log['users.without_list_rows'] = int((~users['has_list_rows']).sum())
value_log['anime.not_yet_aired_with_completed_rows'] = int(anime['not_yet_aired_with_completed_rows'].sum())

row_log_df = pd.DataFrame(row_log)
value_log_df = (pd.DataFrame(value_log.items(), columns=['rule', 'count'])
                  .sort_values('rule').reset_index(drop=True))
row_log_df.to_csv(OUTPUT_DIR / 'row_count_log.csv', index=False)
value_log_df.to_csv(OUTPUT_DIR / 'value_cleaning_log.csv', index=False)

manifest = pd.DataFrame([
    {'file': anime_path.name, 'rows': len(anime), 'purpose': 'cleaned anime table'},
    {'file': users_path.name, 'rows': len(users), 'purpose': 'cleaned user table'},
    {'file': str(LIST_DIR.relative_to(OUTPUT_DIR)), 'rows': clean_list_rows, 'purpose': 'chunked cleaned user-anime table (parquet, clean columns only)'},
    {'file': 'row_count_log.csv', 'rows': len(row_log_df), 'purpose': 'every row-count change'},
    {'file': 'value_cleaning_log.csv', 'rows': len(value_log_df), 'purpose': 'value-level transformations and flags'},
])
manifest.to_csv(OUTPUT_DIR / 'manifest.csv', index=False)

assert len(anime) == 14_478
assert len(users) == 302_673
assert users['username'].is_unique
assert not users['username'].isin(ALIAS_TO_DROP).any()
assert not anime['anime_id'].duplicated().any()

print('\nRow-count log:')
display(row_log_df)
print('\nKey value-level changes:')
display(value_log_df[value_log_df['count'].gt(0)])
print('\nOutput manifest:')
display(manifest)



Row-count log:


,table,rule,before_rows,after_rows,row_delta,reason
0,anime,value corrections only,14478,14478,0,No anime rows are removed; raw values are reta...
1,users,remove duplicate aliases,302675,302673,-2,Io_Otonashi and Dojin_ duplicate Edengarde and...
2,lists,remove duplicate alias rows,80076112,80075053,-1059,The lists for two renamed duplicate accounts a...



Key value-level changes:


,rule,count
0,anime.episodes_zero_to_missing,504
1,anime.low_raters,4789
2,anime.not_yet_aired_with_completed_rows,6
3,anime.popularity_zero_to_missing,9
4,anime.rank_zero_to_missing,9
5,anime.score_zero_to_missing,354
6,anime.text_placeholder_to_missing,7653
7,lists.completed_episode_count_inferred,284297
8,lists.date_full_invalid_to_missing,74325
9,lists.date_year_partial,756482



Output manifest:


,file,rows,purpose
0,AnimeList_clean.csv,14478,cleaned anime table
1,UserList_clean.csv,302673,cleaned user table
2,user_anime_list_parts,80075053,"chunked cleaned user-anime table (parquet, cle..."
3,row_count_log.csv,3,every row-count change
4,value_cleaning_log.csv,26,value-level transformations and flags


## E. Archive the processed dataset for reuse

Kaggle working files should be downloaded or saved as a Kaggle Dataset before the session is discarded. This creates one ZIP containing the complete processed folder, including its logs and manifest. Upload that ZIP as a **private Kaggle Dataset** first; make it public only if the raw dataset licence and your team agree.


In [7]:
# Run after the finalisation cell. The ZIP is a sibling of the processed folder, so it never includes itself.
archive_path = shutil.make_archive(
    base_name=str(PROCESSED_ROOT / VERSION),
    format='zip',
    root_dir=PROCESSED_ROOT,
    base_dir=VERSION,
)
archive_size_gb = Path(archive_path).stat().st_size / 1024**3
print(f'Archive created: {archive_path} ({archive_size_gb:.2f} GB)')
print('Kaggle: download this file or create a private Dataset from it, then attach that Dataset to Steps 5–6.')


Archive created: /kaggle/working/processed/step4_clean_v1.zip (0.80 GB)
Kaggle: download this file or create a private Dataset from it, then attach that Dataset to Steps 5–6.


## What we know but did not do, and why

Step 3 found more problems than this notebook fixes. The list below is on purpose: each item is a problem we **know** about and **decided not to fix here**, with the reason and the place where it can still be handled. The assignment asks us to justify every cleaning decision, and "not worth it" is a decision too.

| Known problem | Why it is not done here | Where it can be handled |
|---|---|---|
| **Bayesian score** (shrinking the score of anime with few raters) | It needs the mean score of all anime, which is a value learned from data. Computed here, it would leak into any train/test split. | In the ML `Pipeline`, fitted on the training set. Raw `score` and the `low_raters` flag are kept. |
| **Imputing missing gender, birth date and location** (28%, 44%, 48% of users) | Imputing learns from the data, and most questions do not need these fields. A guessed value would look real. | Analyse the users who have the field and say so, or impute inside the `Pipeline`. |
| **Capping extreme user totals at a percentile** | A percentile is learned from the data. Only the impossible values (more than 10,000 days, more than 500,000 episodes) become missing here. | `extreme_user` flags the rest. Cap or log-transform in the `Pipeline`, and test another threshold as a robustness check (D6). |
| **Grouping `location` into countries** | We only lowercase and trim. That reduces 55,283 raw spellings to 49,616, and a real grouping needs a lookup table and manual review. | Only if a question is about geography. |
| **The meaning of `undefined`, `anYme` and `*` in `my_tags`** | Only the listed placeholder words are removed. We do not know what the other symbols mean, and removing them would be a guess. | Decide when a question uses tags. `undefined` and similar words are about 53,395 cells. |
| **Storing flags on the list table** (inconsistent episodes, invalid or partial dates) | Each flag can be rebuilt from the raw and clean columns, and one True/False column costs roughly 0.5 GB as CSV over 80M rows. | The counts are in `value_cleaning_log.csv`. Rebuild a flag in the notebook that needs it. |
| **Filling `completed` rows with the anime's episode count** | It rests on one assumption (a completed anime was watched in full) and changes 0.2–0.56% of the completed rows. | In the one question that needs a full-watch count, with its own row count. |
| **Repairing the anime `aired` dates** (373 made-up dates, 71 a year off around New Year) | We avoid the problem instead of repairing it: the year is read from `aired_string`. A repair would need a second source. | Not needed if only the year is used. |
| **Dropping mostly empty anime columns** (`background`, `licensor`, `premiered`, `broadcast`) and `airing` | Dropping saves nothing in a table of 14,478 rows. `airing` is identical to `status`, but keeping it is harmless. | Choose the columns when features are selected. |
| **Splitting `broadcast` into weekday and time** | The column is about 70% empty and unlikely to be used. | Skip, unless a question is about broadcast slots. |
| **Removing low-rater anime, users without list rows (19,630) and users with an empty profile (129)** | These rows are real. Removing them here would decide for every question at once. | Each question sets its own filter and reports the row count before and after (assignment 3.4). |
| **Reconciling declared counts with actual list rows** (0.1–1.8% per status) | The site computes the declared counts, and the difference is probably timing between the profile and the list. It is not an error we can fix. | Use the list rows as the truth. |
| **Checking `stats_episodes` against the list totals** | It needs a groupby per user over 80M rows, and it is an analysis, not a cleaning rule. | Step 5 (05a), re-run on the cleaned data. |
| **Collapsing double spaces in titles** (32 `title_japanese` cells) | Cosmetic. Titles are not used for any analysis. | Only if titles are matched between tables. |

**What this notebook does not prove.** It applies the rules and counts what changed, but it does not show that the cleaned data answers the questions correctly. The meanings tagged [V] and the results computed on raw data are re-checked on the cleaned data in step 5.


## Next step

1. Download or retain the `step4_clean_v1` folder. For a local run, place it under `data/processed/`. Read the list table with `pd.read_parquet(folder)` (all parts), or one part at a time for memory.
2. Update and rerun 05a, 05b and 05c against these outputs. Their raw-data findings cannot become reported results until that re-check is complete.
3. Then formulate two decision-focused project questions. Each may contain several smaller diagnostic questions, but each needs one clear stakeholder decision, a method, quantified result and limitations.
